# Model extraction & stealing

Model extraction learns a substitute model by querying another model and fitting to its outputs.

Gap note: the lesson body is thin, so the notebook grounds the method in the plan and demonstrates query-budget, hard-label, and probability-output stealing. Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.datasets import load_breast_cancer
from sklearn.datasets import make_blobs
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 19
rng = np.random.default_rng(SEED)


def clf_ladder():
    """D1..D5 classification ladder of rising complexity. Returns [(name, X, y), ...]."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def make_group(X, y):
    scores = X[:, 0]
    if len(np.unique(scores)) < 3:
        scores = X.sum(axis=1)
    cutoff = np.median(scores)
    group = (scores > cutoff).astype(int)
    if len(np.unique(group)) < 2:
        group = (np.arange(len(y)) % 2).astype(int)
    return group


def safe_split(X, y, group=None, test_size=0.4):
    stratify = y
    if min(np.bincount(y.astype(int))) < 2:
        stratify = None
    pieces = train_test_split(
        X,
        y,
        np.arange(len(y)),
        test_size=test_size,
        random_state=0,
        stratify=stratify,
    )
    x_train, x_test, y_train, y_test, idx_train, idx_test = pieces
    if group is None:
        return x_train, x_test, y_train, y_test, idx_train, idx_test, None, None
    return x_train, x_test, y_train, y_test, idx_train, idx_test, group[idx_train], group[idx_test]


def fit_scaled_logreg(X, y, C=1.0):
    x_train, x_test, y_train, y_test, idx_train, idx_test, _, _ = safe_split(X, y)
    scaler = StandardScaler()
    x_train_s = scaler.fit_transform(x_train)
    x_test_s = scaler.transform(x_test)
    model = LogisticRegression(max_iter=2000, C=C, multi_class="auto")
    model.fit(x_train_s, y_train)
    return model, scaler, x_train_s, x_test_s, y_train, y_test, idx_train, idx_test


def probability_for_label(model, X, y):
    probs = model.predict_proba(X)
    positions = np.array([np.where(model.classes_ == label)[0][0] for label in y])
    return probs[np.arange(len(y)), positions]


def fgsm_attack(model, X, y, eps):
    probs = model.predict_proba(X)
    class_positions = np.array([np.where(model.classes_ == label)[0][0] for label in y])
    one_hot = np.zeros_like(probs)
    one_hot[np.arange(len(y)), class_positions] = 1.0
    grad = (probs - one_hot) @ model.coef_
    direction = np.sign(grad)
    return X + eps * direction


def robust_accuracy_for_eps(model, X, y, eps):
    attacked = fgsm_attack(model, X, y, eps)
    preds = model.predict(attacked)
    return accuracy_score(y, preds)


def fairness_report(y, yhat, group):
    rows = {}
    positive = int(np.max(y))
    for g in [0, 1]:
        mask = group == g
        truth_pos = mask & (y == positive)
        truth_neg = mask & (y != positive)
        pred_pos = mask & (yhat == positive)
        tp = int(np.sum(pred_pos & truth_pos))
        fp = int(np.sum(pred_pos & truth_neg))
        fn = int(np.sum((~pred_pos) & truth_pos))
        tn = int(np.sum((~pred_pos) & truth_neg))
        rate = float(np.mean(yhat[mask] == positive)) if np.any(mask) else np.nan
        tpr = tp / max(tp + fn, 1)
        fpr = fp / max(fp + tn, 1)
        rows[g] = {"n": int(mask.sum()), "pos_rate": rate, "tpr": tpr, "fpr": fpr, "tp": tp, "fp": fp, "fn": fn, "tn": tn}
    dp_gap = abs(rows[0]["pos_rate"] - rows[1]["pos_rate"])
    eo_gap = max(abs(rows[0]["tpr"] - rows[1]["tpr"]), abs(rows[0]["fpr"] - rows[1]["fpr"]))
    return {"group0": rows[0], "group1": rows[1], "dp_gap": dp_gap, "eo_gap": eo_gap}


def plot_2d_projection(ax, X, color, title):
    if X.shape[1] >= 2:
        shown = X[:, :2]
    else:
        shown = np.column_stack([X[:, 0], np.zeros(len(X))])
    ax.scatter(shown[:, 0], shown[:, 1], c=color, s=18, cmap="viridis", alpha=0.75)
    ax.set_title(title, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])


## The concept, built once: substitute training

The extraction objective is $$\hat g=\arg\min_g \frac1q\sum_{i=1}^{q}\ell(g(x_i),f(x_i)).$$ Gap note: the lesson body is thin, so the plan's stable toy numbers anchor the assertions.

In [ ]:

components = np.array([0.9, 0.6, 0.3], dtype=float)
knob = 0.15
total = float(np.sum(components))
absolute_total = float(np.sum(np.abs(components)))
leading_share = float(abs(components[0]) / absolute_total)
guarded = float(total + knob * absolute_total)
contrast = float(total - components[2])
change = float(contrast - total)

assert np.isclose(total, 1.8)
assert np.isclose(absolute_total, 1.8)
assert np.isclose(round(guarded, 3), 2.07)
assert np.isclose(round(leading_share, 3), 0.5)
assert np.isclose(round(leading_share, 3), 0.500)

print("total", round(total, 3))
print("absolute_total", round(absolute_total, 3))
print("leading_share", round(leading_share, 3))
print("guarded", round(guarded, 3))
print("change_without_component_3", round(change, 3))


A real extraction attack queries a teacher on chosen inputs, then trains a student on hard labels or probabilities. Agreement, not parameter recovery, is the risk metric.

In [ ]:

def extract_model(teacher, X_pool, query_strategy="random", q=50, use_probabilities=False):
    if query_strategy == "boundary":
        probs = teacher.predict_proba(X_pool)
        uncertainty = 1.0 - np.max(probs, axis=1)
        chosen = np.argsort(uncertainty)[-q:]
    else:
        chosen = np.linspace(0, len(X_pool) - 1, min(q, len(X_pool))).astype(int)
    query_x = X_pool[chosen]
    if use_probabilities:
        soft = teacher.predict_proba(query_x)
        labels = teacher.classes_[np.argmax(soft, axis=1)]
    else:
        labels = teacher.predict(query_x)
    student = LogisticRegression(max_iter=2000, multi_class="auto")
    student.fit(query_x, labels)
    return student, chosen

X, y = clf_ladder()[1][1:]
model, scaler, x_train, x_test, y_train, y_test, idx_train, idx_test = fit_scaled_logreg(X, y)
student, chosen = extract_model(model, x_train, query_strategy="boundary", q=40)
agreement = accuracy_score(model.predict(x_test), student.predict(x_test))
print("queries", len(chosen))
print("teacher_student_agreement", round(agreement, 3))


## The dataset ladder

The same classifier family is tested on D1-D5: a hand toy, synthetic blobs, noisy moons, real Wine data, and real Breast Cancer data.

In [ ]:

rungs = clf_ladder()
for name, X, y in rungs:
    classes, counts = np.unique(y, return_counts=True)
    print(name)
    print("  shape", X.shape)
    print("  classes", dict(zip(classes.tolist(), counts.tolist())))
    print("  sample", np.round(X[:3, :min(4, X.shape[1])], 3))


## Run the same extraction method across D1-D5

The metric is teacher-student agreement on held-out examples for a fixed query budget.

In [ ]:

query_budget = 80
extract_results = []
for rung, (name, X, y) in enumerate(clf_ladder(), start=1):
    model, scaler, x_train, x_test, y_train, y_test, idx_train, idx_test = fit_scaled_logreg(X, y)
    budget = min(query_budget, len(x_train))
    student, chosen = extract_model(model, x_train, query_strategy="boundary", q=budget)
    agreement = accuracy_score(model.predict(x_test), student.predict(x_test))
    extract_results.append({"rung": rung, "name": name, "agreement": agreement, "queries": budget})
print("rung | queries | teacher_student_agreement")
for row in extract_results:
    print(row["rung"], row["queries"], round(row["agreement"], 3))


## Results visualization

Left: teacher versus substitute predictions. Right: agreement versus query budget.

In [ ]:

fig, axes = plt.subplots(1, 5, figsize=(16, 3))
for ax, row, data in zip(axes, extract_results, clf_ladder()):
    name, X, y = data
    model, scaler, x_train, x_test, y_train, y_test, idx_train, idx_test = fit_scaled_logreg(X, y)
    student, chosen = extract_model(model, x_train, query_strategy="boundary", q=min(row["queries"], len(x_train)))
    plot_2d_projection(ax, x_test, student.predict(x_test), f"D{row['rung']} student")
plt.tight_layout()
plt.show()

budgets = [10, 25, 50, 100, 200, 500]
fig, ax = plt.subplots(figsize=(7, 4))
for rung, (name, X, y) in enumerate(clf_ladder(), start=1):
    model, scaler, x_train, x_test, y_train, y_test, idx_train, idx_test = fit_scaled_logreg(X, y)
    curve = []
    for budget in budgets:
        q = min(budget, len(x_train))
        student, chosen = extract_model(model, x_train, query_strategy="boundary", q=q)
        curve.append(accuracy_score(model.predict(x_test), student.predict(x_test)))
    ax.plot(budgets, curve, marker="o", label=f"D{rung}")
ax.set_xlabel("query budget")
ax.set_ylabel("teacher-student agreement")
ax.legend()
plt.show()


## Pitfall on D5: ignoring confidence outputs

The wrong defense evaluates only hard labels. The fix compares hard-label and probability-exposed APIs, then adds a simple diversity check for adaptive querying.

In [ ]:

name, X, y = clf_ladder()[-1]
model, scaler, x_train, x_test, y_train, y_test, idx_train, idx_test = fit_scaled_logreg(X, y)
hard_student, hard_chosen = extract_model(model, x_train, query_strategy="random", q=120, use_probabilities=False)
prob_student, prob_chosen = extract_model(model, x_train, query_strategy="boundary", q=120, use_probabilities=True)
hard_agreement = accuracy_score(model.predict(x_test), hard_student.predict(x_test))
prob_agreement = accuracy_score(model.predict(x_test), prob_student.predict(x_test))
query_spread = np.mean(np.std(x_train[prob_chosen], axis=0))
print("wrong_hard_label_only_agreement", round(hard_agreement, 3))
print("fixed_probability_boundary_agreement", round(prob_agreement, 3))
print("diversity_spread_for_rate_limit", round(float(query_spread), 3))


## Evaluate it + Practice

- Metric: teacher-student agreement.
- No-skill baseline: student trained on random labels or tiny random query set.
- Cheap sanity check: agreement should increase with query budget on easier rungs.
- Ablation: use random interior queries instead of boundary queries.
- Failure signals: rate limits ignore query diversity or confidence-output leakage.

Practice prompts:
1. Change one stress knob and predict the direction of the metric before running.


2. Compare random and boundary query strategies at the same budget.

3. Add a watermark slice and measure whether the student copied it.